In [1]:
from pathlib import Path
from typing import Iterable
import re
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup



def _sanitize_filename(name: str) -> str:
    name = re.sub(r'[<>:"/\\|?*\n\r\t]+', " ", name).strip()
    name = re.sub(r"\s+", "_", name)
    return name or "tale"


def _iter_links_from_index(base_url: str, index_html: str) -> Iterable[tuple[str, str]]:
    soup = BeautifulSoup(index_html, "html.parser")
    contents = soup.find("ol") or soup.find("ul")
    if not contents:
        return []
    links = []
    for a in contents.find_all("a", href=True):
        href = a["href"]
        text = a.get_text(strip=True)
        # skip obvious directory entries and external links
        if href in ("../", "/", "#"):
            continue
        if href.startswith("http") and not href.startswith(base_url):
            continue
        if href.endswith((".html", ".htm", ".txt")) or href.endswith("/") or "." not in href:
            links.append((href, text))
    return links


def _extract_text_from_page(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    # prefer article or main, then paragraphs, then full text
    for selector in ("article", "main", "div#content", "div.article", "body"):
        node = soup.find(selector)
        if node:
            text = node.get_text(separator="\n").strip()
            if text:
                return text
    return soup.get_text(separator="\n").strip()


def generate_fairy_tale_files(
    base_url: str = "https://www.cs.cmu.edu/~spok/grimmtmp/",
    output_dir: str = "resources/fairy-tales",
) -> None:
    """Download pages listed on the given index and write text files to output_dir.

    The index page is expected to contain an ordered or unordered list of links
    to individual tale pages (html or txt). The function will follow relative
    links and save each tale as a UTF-8 text file.
    """
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    resp = requests.get(base_url)
    resp.raise_for_status()
    links = _iter_links_from_index(base_url, resp.text)

    if not links:
        # If no links found, try to treat base_url as a page list page by scanning all anchors
        soup = BeautifulSoup(resp.text, "html.parser")
        anchors = [(a.get('href', ''), a.get_text(strip=True)) for a in soup.find_all('a', href=True)]
        links = anchors

    for href, link_text in links:
        tale_url = urljoin(base_url, href)
        try:
            tale_resp = requests.get(tale_url)
            tale_resp.raise_for_status()
        except Exception:
            # skip failed fetches
            continue

        title = link_text or href
        # attempt to extract a better title from the page
        try:
            t_soup = BeautifulSoup(tale_resp.text, "html.parser")
            title_tag = t_soup.find("h1") or t_soup.find("title")
            if title_tag and title_tag.get_text(strip=True):
                title = title_tag.get_text(strip=True)
        except Exception:
            pass

        filename = _sanitize_filename(title) + ".txt"
        text = ""
        # if it's a text file link, use raw content
        if tale_url.lower().endswith(".txt"):
            text = tale_resp.text
        else:
            text = _extract_text_from_page(tale_resp.text)

        file_path = out_path / filename
        file_path.write_text(text, encoding="utf-8")
        print(f"Saved {title} -> {file_path}")


In [2]:
generate_fairy_tale_files()

Saved The Frog King, or Iron Henry -> resources\fairy-tales\The_Frog_King,_or_Iron_Henry.txt
Saved Our Lady's Child -> resources\fairy-tales\Our_Lady's_Child.txt
Saved The Story of a Youth Who Went Forth to Learn What Fear Was -> resources\fairy-tales\The_Story_of_a_Youth_Who_Went_Forth_to_Learn_What_Fear_Was.txt
Saved The Wolf and the Seven Little Kids -> resources\fairy-tales\The_Wolf_and_the_Seven_Little_Kids.txt
Saved Faithful John -> resources\fairy-tales\Faithful_John.txt
Saved The Good Bargain -> resources\fairy-tales\The_Good_Bargain.txt
Saved The Twelve Brothers -> resources\fairy-tales\The_Twelve_Brothers.txt
Saved Brother and Sister -> resources\fairy-tales\Brother_and_Sister.txt
Saved Rapunzel -> resources\fairy-tales\Rapunzel.txt
Saved The Three Little Men in the Wood -> resources\fairy-tales\The_Three_Little_Men_in_the_Wood.txt
Saved The Three Spinners -> resources\fairy-tales\The_Three_Spinners.txt
Saved Hansel and Grethel -> resources\fairy-tales\Hansel_and_Grethel.txt


In [3]:
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.runnables import RunnableLambda
from langchain_core.documents import Document

load_dotenv(override=True)

candidate_paths = [
    Path("resources/fairy-tales"),
    Path("../resources/fairy-tales"),
    Path.cwd() / "resources" / "fairy-tales"
]
tale_dir = next((p for p in candidate_paths if p.exists()), None)
assert tale_dir is not None, f"Fairy tale directory not found. Checked: {candidate_paths}"
repo_root = tale_dir.parent.parent
print(f"Using fairy tale directory: {tale_dir.resolve()}")
print(f"Repository root seems to be: {repo_root.resolve()}")

Using fairy tale directory: C:\Users\scott\projects\llm_engineering\my-stuff\rag-fairy-tales\resources\fairy-tales
Repository root seems to be: C:\Users\scott\projects\llm_engineering\my-stuff\rag-fairy-tales


In [4]:
file_paths = sorted(tale_dir.glob("*.txt"))
print(f"Found {len(file_paths)} fairy tale files")

fairy_tales = []
for path in file_paths:
    text = path.read_text(encoding="utf-8")
    title = path.stem.replace("_", " ")
    fairy_tales.append({"title": title, "path": str(path), "text": text})

print("Loaded fairy tales and metadata. Sample title:", fairy_tales[0]["title"] if fairy_tales else "none")

Found 208 fairy tale files
Loaded fairy tales and metadata. Sample title: A Riddling Tale


In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunk_docs = []
for tale in fairy_tales:
    chunks = splitter.split_text(tale["text"])
    for i, chunk in enumerate(chunks):
        chunk_docs.append(Document(page_content=chunk, metadata={
            "title": tale["title"],
            "source": tale["path"],
            "chunk": i,
        }))

print(f"Created {len(chunk_docs)} text chunks for embedding")

embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")
vector_db_dir = repo_root / "resources" / "fairy_tales_vector_db"
vector_db = Chroma.from_documents(
    documents=chunk_docs,
    embedding=embedding_model,
    persist_directory=str(vector_db_dir),
    collection_name="fairy_tales",
)
print(f"Built or loaded Chroma vector store at: {vector_db_dir.resolve()}")

Created 2018 text chunks for embedding
Built or loaded Chroma vector store at: C:\Users\scott\projects\llm_engineering\my-stuff\rag-fairy-tales\resources\fairy_tales_vector_db


In [6]:
llm = ChatOpenAI(model_name="gpt-4.1-nano", temperature=0)
retriever = vector_db.as_retriever(search_kwargs={"k": 4})

def make_answer_chain(question: str):
    def docs_to_messages(docs):
        if not docs:
            return [
                SystemMessage(
                    "You are an expert assistant answering questions about Grimm fairy tales. Use only the retrieved context to answer the user clearly and accurately. If the answer is not present in the context, say you don't know."
                ),
                HumanMessage(content=f"Question: {question}\n\nContext:\nNo relevant context was found."),
            ]

        context = "\n\n".join(
            f"Source: {doc.metadata.get('title', doc.metadata.get('source', 'unknown'))}\n{doc.page_content}" for doc in docs
        )
        return [
            SystemMessage(
                "You are an expert assistant answering questions about Grimm fairy tales. Use only the retrieved context to answer the user clearly and accurately. If the answer is not present in the context, say you don't know."
            ),
            HumanMessage(content=f"Question: {question}\n\nContext:\n{context}"),
        ]

    return retriever | RunnableLambda(docs_to_messages) | llm


def answer_fairy_tales_question(question: str) -> str:
    chain = make_answer_chain(question)
    response = chain.invoke(question)
    return response.content

print(answer_fairy_tales_question("What gift does Cinderella lose at the ball?"))

Cinderella loses her glass slipper (referred to as a golden shoe in the story) at the ball.


## Gradio Fairy Tale QA Interface

Ask a question about Grimm fairy tales and let the retriever-augmented model answer using the fairy tale document store.

In [7]:
import gradio as gr

def gradio_answer(question: str) -> str:
    return answer_fairy_tales_question(question)

qa_interface = gr.Interface(
    fn=gradio_answer,
    inputs=gr.Textbox(lines=2, placeholder="Ask a question about Grimm fairy tales...", label="Question"),
    outputs=gr.Textbox(label="Answer", lines=20),
    title="Grimm Fairy Tales QA",
    description="Retriever-augmented question answering over the fairy tale texts.",
    allow_flagging="never",
)

qa_interface.launch()

c:\Users\scott\projects\llm_engineering\.venv\Lib\site-packages\gradio\interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
